# Day 26 — Tool-Using Agents: a ReAct Agent over Calculator, Web Search & RAG

**Sehar Andleeb — AI Engineer Intern, Xeven Solutions**

Today's roadmap topic is agentic tool use: giving an LLM a small set of
tools and letting it *decide* which one to call, in what order, based
on the question. Rather than a single-purpose script, I built a
general-purpose **ReAct (Reason + Act) agent** that routes between
three tools — a safe calculator, a live web search, and the Day 25
ScholarRAG retrieval pipeline — and wrapped it in a small FastAPI + JS
chat UI so it's actually usable, not just a notebook demo.


![Day 26 chat UI, showing a question routed to the calculator tool and an answer](screenshots/chat-ui.png)


## Concepts

**ReAct (Reason + Act).** Instead of answering directly, the LLM is
prompted to alternate between a `Thought` (reasoning about what to do
next), an `Action` (picking a tool and an input for it), and an
`Observation` (the tool's real output), looping until it has enough
information to give a `Final Answer`. This lets a single model handle
arithmetic, current events, and domain-specific retrieval without
hand-written routing logic — the model itself decides which tool fits
the question.

**Tool design.** Each tool is a small, single-purpose function wrapped
with LangChain's `@tool` decorator, which turns a plain Python function
into something the agent can call by name with a string input. The
docstring *is* the tool's interface to the model — it's the only thing
the LLM sees when deciding whether and how to use the tool, so it has
to be precise about expected input format.

**Safety in the calculator.** A naive `eval()`-based calculator is a
code-execution vulnerability — anything that can reach `eval` can also
reach `__import__('os').system(...)`. The calculator tool instead walks
a restricted `ast` parse tree and only allows numeric literals and
arithmetic operators, rejecting anything else (function calls, names,
attribute access) before it can be evaluated.

**Tool surface vs. implementation.** Calling a `@tool`-wrapped function
directly (`my_tool(x)`) is deprecated in recent LangChain — tools are
`BaseTool` objects now, not plain callables, so the correct call is
`my_tool.invoke(x)`. This matters once tools are composed (e.g. one
tool internally reusing another), which is the case here, since the
agent's tool wrappers call into the underlying `calculator`,
`web_search`, and `rag_search` tools from `tools/`.


## Research: design questions and tradeoffs

| # | Question | Alternative approach | What I implemented & why |
|---|----------|----------------------|---------------------------|
| 1 | How should the agent decide which tool to call? | A hand-written intent classifier (regex or a small trained model) routes the query to one fixed tool before any LLM call. | LangChain's `create_react_agent` with a ReAct prompt — the LLM reasons about tool choice itself, can chain multiple tool calls per question, and self-corrects on a bad call (seen live: the calculator tool failed on a quoted input, and the agent retried with the un-quoted form on its own). |
| 2 | How do I stop the calculator from being an arbitrary code-execution hole? | Use Python's built-in `eval()` directly, optionally with a denylist of dangerous names. | Parse the expression with `ast.parse` and walk the tree, allowing only `Constant`, `BinOp`, and arithmetic operators. A `__import__('os').system(...)` payload is rejected at the parse-walk stage, not the execution stage, since it never matches an allowed node type. |
| 3 | Which web search backend to use? | Paid APIs (SerpAPI, Tavily) give more reliable structured results. | `ddgs` (the renamed `duckduckgo-search` package) — no API key needed, good enough result quality for a demo agent, and it's the same idea as Day 25's "no extra paid dependency" approach. |
| 4 | How does the agent's RAG tool relate to Day 25's RagService? | Re-implement retrieval logic inside the tool. | Reuse Day 25's `RagService` (hybrid retrieval + reranking) as-is from inside `rag_tool.py` — the tool is a thin wrapper that calls `RagService.search()`/`.ask()` and formats the result as a string for the LLM, so all of Day 25's hybrid search + reranking quality carries over for free. |
| 5 | How to expose the agent beyond a script? | A notebook-only demo, or a heavier framework (Streamlit/Gradio). | Plain FastAPI + static HTML/JS, matching Day 25's UI pattern — a `/ask` POST endpoint wraps `AgentExecutor.invoke()`, and a small chat-style frontend (avatars, typing indicator, animated bubbles) calls it via `fetch`. |


## Executed: each tool in isolation

Before wiring tools into the agent, each one was tested standalone to
confirm correct behavior — including that the calculator safely
rejects code-injection attempts.


In [1]:
"""Path setup: the day26/tools/ modules aren't on sys.path by default."""
import sys

if "tools" not in sys.path:
    sys.path.insert(0, ".")


In [2]:
"""Calculator tool: arithmetic works, division-by-zero and code
injection are both safely rejected."""
from tools.calculator_tool import calculator

print(calculator.invoke("23 * 45"))
print(calculator.invoke("10 / 0"))
print(calculator.invoke("__import__('os').system('echo hacked')"))


1035
Error: division by zero.
Error: could not evaluate '__import__('os').system('echo hacked')' (Unsupported expression node: Call(func=Attribute(value=Call(func=Name(id='__import__', ctx=Load()), args=[Constant(value='os')], keywords=[]), attr='system', ctx=Load()), args=[Constant(value='echo hacked')], keywords=[])).


In [3]:
"""Web search tool: live DuckDuckGo results via ddgs."""
from tools.web_search_tool import web_search

print(web_search.invoke("LangChain ReAct agent"))


Top web results for 'LangChain ReAct agent':
1. create_react_agent | langchain_classic | LangChain Reference
   This implementation is based on the foundational ReAct paper but is older and not well-suited for production applications. For a more robust and feature-rich implementation, we recommend using the create_agent function from the langchain library.
   Source: https://reference.langchain.com/python/langchain-classic/agents/react/agent/create_react_agent
2. GitHub - langchain-ai/react-agent: LangGraph template for a simple ReAct agent · GitHub
   The core logic, defined in src/react_agent/graph.py, demonstrates a flexible ReAct agent that iteratively reasons about user queries and executes actions, showcasing the power of this approach for complex problem-solving tasks.
   Source: https://github.com/langchain-ai/react-agent
3. LangChain ReAct Agent: Complete Implementation Guide + Working Examples 2025 - Latenode Blog
   May 12, 2026 - The LangChain ReAct Agent is a problem-solvi

In [4]:
"""RAG tool: reuses Day 25's RagService over the ingested
"Attention Is All You Need" paper."""
from tools.rag_tool import rag_search

print(rag_search.invoke("What is self-attention?"))


Top document excerpts for 'What is self-attention?':
1. [Attention Is All You Need - 2 Background] (score=9.000)
   an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent ...
2. [Attention Is All You Need - 2 Background] (score=9.000)
   The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [ 16 ] , ByteNet [ 18 ] and ConvS2S [ 9 ] , all of which use convolutional neural networks as basic building block, computing hidden representations in parallel for all input and output positions. In the...
3. [Attention Is All You Need - 3 Model Architecture] (score=9.000)
   mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed a

## Live pipeline: the full ReAct agent

The cell below needs `GROQ_API_KEY` in `.env` and a live network
connection (Groq + DuckDuckGo + the local RAG index), so it doesn't
execute inside this notebook-building environment. The code is
identical to `agent.py` in `day26/`; the output shown is the real
output from running it on my own machine, asking one question per
tool to confirm correct routing.


In [5]:
"""Full ReAct agent: ask one question per tool, watch it route and
self-correct."""
from agent import build_agent

executor = build_agent()

for q in [
    "What is 23 * 45?",
    "What is self-attention according to the paper?",
    "What's the latest news on LangChain agents?",
]:
    result = executor.invoke({"input": q})
    print("Q:", q)
    print("A:", result["output"])
    print()


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: To find the product of 23 and 45, I can use the calculator tool to evaluate the math expression.

Action: calculator
Action Input: 23 * 451035Thought: I have successfully evaluated the math expression using the calculator tool, and I now have the result.

Final Answer: 1035

> Finished chain.
Q: What is 23 * 45?
A: 1035

Thought: To answer this question, I need to find information about self-attention from a reliable source, specifically a paper. Since I have access to the RAG index, which contains ingested arXiv papers, I can use this tool to search for the definition of self-attention.

Action: rag_tool
Action Input: self-attention paperTop document excerpts for 'self-attention paper':
1. [Attention Is All You Need - 4 Why Self-Attention] (score=10.000)
   separable convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer, the approach we take in our model. As side benefit, self-attention could yield more interpretable models. We 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: Based on the observation from the RAG index, it seems that self-attention is a mechanism used in sequence-to-sequence models, where the attention is applied to the same sequence, allowing the model to relate different positions of the sequence to compute a representation of the sequence. This is mentioned in the paper "Attention Is All You Need". The self-attention layer is used in the encoder, where all the keys, values, and queries come from the same place, which is the output of the previous layer in the encoder. I can now provide a definition of self-attention based on this information.

Final Answer: Self-attention is a mechanism used in sequence-to-sequence models that relates different positions of a single sequence to compute a representation of the sequence, where all the keys, values, and queries come from the same place, allowing the model to attend to all positions in the sequence simultaneously.

> Finished chain.
Q: What is self-attention according to the paper?


## Serving it: FastAPI + chat UI

`ui_main.py` wraps the agent in a single `/ask` POST endpoint, built
once at startup (`lifespan`) so the agent isn't rebuilt per-request,
and serves `ui/index.html` as static files — same pattern as Day 25's
`main.py`. The frontend is a small chat interface: user/bot message
bubbles, an animated typing indicator while the agent is reasoning,
and a gradient header, built in vanilla HTML/CSS/JS with no framework
dependency.


![Day 26 chat UI: gradient header, chat bubbles, typing indicator](screenshots/chat-ui-full.png)


## Architecture summary

- **Agent** — LangChain `create_react_agent` + `AgentExecutor`, Groq `llama-3.3-70b-versatile`, `temperature=0`, `max_iterations=6`, `handle_parsing_errors=True`
- **Calculator tool** — `ast`-based safe expression evaluator; rejects any node that isn't a numeric literal or arithmetic operator
- **Web search tool** — `ddgs` (DuckDuckGo), formatted top-N results with title/snippet/source
- **RAG tool** — thin wrapper around Day 25's `RagService` (hybrid FAISS + BM25 retrieval, Groq reranking) over the ingested "Attention Is All You Need" paper
- **Serving** — FastAPI `/ask` endpoint, agent built once at startup via `lifespan`, static files mounted for the UI
- **UI** — vanilla HTML/CSS/JS chat interface, no framework, talks to `/ask` via `fetch`

## What I'd improve with more time

- Stream the agent's intermediate `Thought`/`Action` steps to the UI live, instead of only showing the final answer after the full loop completes
- Add a short system-level instruction to the calculator tool's docstring telling the model not to wrap input in quotes, to remove the one observed self-correction step
- Persist conversation history server-side so the agent can answer follow-up questions with context from earlier turns in the same session
